In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/home/shared/data/imagenet/validation-*.parquet"))
print(files)  # sanity check

df = pd.read_parquet(files)

In [ ]:

print(len(df))


In [ ]:
import numpy as np

SEED = 42
TEST_SAMPLES_PER_CLASS = 32

# Shuffle within each class, then split
test_df = (
    df.groupby("label", group_keys=False)
      .apply(lambda x: x.sample(n=TEST_SAMPLES_PER_CLASS, random_state=SEED))
)

train_df = df.drop(test_df.index)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


In [ ]:
test_df["label"].value_counts()


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import torch
from torchvision import models

weights = models.ResNet50_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
preprocess

In [ ]:
dataset = ImageNetParquetDataset(
    df,
    transform=preprocess
)

In [ ]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

weights = models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)

# Remove classification head
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()


In [ ]:
from PIL import Image
import io
import matplotlib.pyplot as plt

# Get the image bytes
img_bytes = df.loc[10, 'image']['bytes']

# Convert bytes → PIL Image
img = Image.open(io.BytesIO(img_bytes))

# Plot
plt.imshow(img)
plt.axis("off")
plt.show()


---

In [ ]:
train_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/train.pt"))
val_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/validation.pt"))
test_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/test.pt"))

In [ ]:
def plot_images(imgs):
    n = len(imgs)
    cols = 5
    rows = int(np.ceil(n / cols))
    
    plt.figure(figsize=(cols * 3, rows * 3))
    
    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        
        # Handle grayscale vs RGB
        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
len(val_dataset)

In [ ]:
plot_images(train_dataset[random.randint(0, 70)][0])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random    

class SingleDigitPadded(Dataset):
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        img, label = self.mnist[idx]

        # Decide randomly whether the digit is on the left or right
        if random.random() < 0.5:
            # digit on left
            empty = torch.zeros_like(img)
            image = torch.cat((img, empty), dim=2)  # 28x56
            position = "left"
        else:
            # digit on right
            empty = torch.zeros_like(img)
            image = torch.cat((empty, img), dim=2)
            position = "right"

        return image, label, position
    

class TwoDigitOpposite(Dataset):
    def __init__(self, padded_dataset):
        """
        padded_dataset: your SingleDigitPadded dataset
        """
        self.images = padded_dataset["images"]
        self.positions = padded_dataset["positions"]
        self.labels = padded_dataset["labels"]

        # Precompute indices by position for faster sampling
        self.left_indices = [i for i in range(len(self.images)) if self.positions[i] == "left"]
        self.right_indices = [i for i in range(len(self.images)) if self.positions[i] == "right"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # First sample (fixed)
        img1, label1, pos1 = self.images[idx], self.labels[idx], self.positions[idx]

        # Determine opposite position
        opposite_pos = "right" if pos1 == "left" else "left"
        candidate_indices = self.right_indices if opposite_pos == "right" else self.left_indices

        # Randomly select second sample with opposite position
        random.seed(42)
        idx2 = random.choice(candidate_indices)
        img2, label2 = self.images[idx2], self.labels[idx2]

        # Combine images by adding (keep same shape)
        two_digit_img = img1 + img2
        two_digit_label = (label1, label2) if pos1 == "left" else (label2, label1)

        return two_digit_img, two_digit_label

    
def create_mnist1():
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    train_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=False,
        download=True,
        transform=transform
    )


    singleDigit_train = SingleDigitPadded(train_dataset)
    singleDigit_test = SingleDigitPadded(test_dataset)

    train_loader = DataLoader(singleDigit_train, batch_size=len(singleDigit_train))  
    test_loader = DataLoader(singleDigit_test, batch_size=len(singleDigit_test)) 

    for images, labels, positions in train_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions  
        }, '/home/shared/data/MNIST2/train.pt')
        break  # only one batch needed

    for images, labels, positions in test_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions
        }, '/home/shared/data/MNIST2/test.pt')
        break  # only one batch needed


def create_mnist2():
    singleDigit = torch.load('/home/shared/data/MNIST1/test.pt')
    
    two_digit_dataset = TwoDigitOpposite(singleDigit)

    # Save the dataset
    loader = DataLoader(two_digit_dataset, batch_size=len(two_digit_dataset))

    for images, labels in loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels[0] * 10 + labels[1],
        }, '/home/shared/data/MNIST2/test.pt')
        break

In [ ]:
create_mnist2()

In [ ]:
import torch
singleDigit = torch.load('/home/shared/data/MNIST2/test.pt')

In [ ]:
import matplotlib.pyplot as plt
import random
i = random.randint(0, 2000)
plt.imshow(singleDigit['images'][i].permute(1, 2, 0), cmap='gray')
plt.title(singleDigit['labels'][i])
plt.axis('off')
plt.show()

# Analysis

In [ ]:
from itertools import combinations, chain, product
import random
from collections import defaultdict
from torch.utils.data import random_split
from functools import reduce
import operator


def SolveMinSym(target_image, all_images):
    """
    Minimum number of positions required to uniquely
    identify the target image.
    """
    distracting_images = [
        img for img in all_images if img != target_image
    ]

    for combination in attribute_combinations(target_image):
        if is_unique_combination(combination, target_image, distracting_images):
            return len(combination)

    return None


def attribute_combinations(image):
    """
    Generate combinations of attribute INDICES
    (not values).
    """
    indices = list(range(len(image)))

    return chain.from_iterable(
        combinations(indices, r)
        for r in range(1, len(indices) + 1)
    )


def is_unique_combination(combination, target_image, distracting_images):
    """
    Check if selected positions uniquely identify target.
    """
    for image in distracting_images:
        match = all(
            image[idx] == target_image[idx]
            for idx in combination
        )

        if match:
            return False

    return True


def min_m_controlled_sampling(dataset, number_of_samples, target_min_symbol, batch_size):

    result = []
    while (len(result) * batch_size) < number_of_samples:

        batch = random.sample(dataset, batch_size)
        target = random.sample(batch, 1)[0]
        
        batch_indices = [image[1] for image in batch]
        batch_values = [image[0] for image in batch]
        target_value, target_index = target
        
        min_symbol = SolveMinSym(target_value, batch_values)
        
        if min_symbol == target_min_symbol:
            result.append({target_value: batch_values})

    return reduce(operator.or_, result)

In [ ]:

from torch.utils.data import Dataset
import torch
import torch.nn.functional as F

        
class ObjectsDataset(Dataset):
    def __init__(self, num_attributes=4, num_values=10, indices=None, min_symbol=2, batch_size=32):
        self.num_attributes = num_attributes
        self.num_values = num_values
        self.data = dict(enumerate(product(range(num_values), repeat=num_attributes)))
        
        if indices is None:
            indices = list(self.data.keys())
        
        self.data = {i: self.data[k] for i, k in enumerate(indices)}
        
        self.candidates = min_m_controlled_sampling(
            [(self.data[k], k) for k in self.data.keys()],
            number_of_samples=len(self.data),
            target_min_symbol=min_symbol,
            batch_size=batch_size
        )

        self.samples = []
        for target, cand in self.candidates:
            self.samples.append(cand, cand.index(target))
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.__to_one_hot(torch.tensor(self.data[idx])), idx
    
    def get_candidates(self):
        return self.candidates
    
    def __to_one_hot(self, x):
        one_hot = F.one_hot(x, num_classes=self.num_values)
        return one_hot.flatten().float()  
    

train, val, test = random_split(range(10000), [0.8, 0.1, 0.1])

train_dataset = ObjectsDataset(indices=train.indices)
val_dataset = ObjectsDataset(indices=val.indices)
test_dataset = ObjectsDataset(indices=test.indices)


In [9]:
from torch.utils.data import DataLoader
import torch

loader = DataLoader(train_dataset, batch_size=len(train_dataset))

In [1]:
l = [1,2 ,3]
l.index(3)

2

In [5]:
for target, labels in loader:
    print(len(target))
    print(len(labels))
    print(target[0])
    print(labels[0])
    break

8000
8000
tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 0.])
tensor(0)


In [ ]:

              # shape: [40]

x = torch.tensor([3, 0, 9, 2])
encoded = encode_one_hit(x)

print(encoded.shape)  # torch.Size([40])


torch.Size([40])


In [88]:
for target, batch in test_dataset:
    print("Target:", target)
    print("Batch:", batch)
    # break

Target: 4505
Batch: [2036, 7029, 4766, 4987, 2138, 4241, 2600, 8612, 4707, 456, 3767, 9291, 7086, 4505, 5380, 8512, 941, 6869, 6811, 207, 2295, 2196, 6499, 3703, 1943, 1900, 2534, 5332, 8004, 3678, 947, 6194]
Target: 6574
Batch: [7312, 6574, 8035, 7222, 5257, 7978, 2796, 2654, 1587, 4082, 3638, 3035, 1360, 4049, 6281, 3535, 9819, 6080, 8384, 5261, 2716, 7019, 764, 2664, 1423, 5626, 1612, 3, 6144, 3665, 1831, 5819]
Target: 1900
Batch: [8230, 7345, 9488, 9397, 1900, 7959, 1094, 7339, 7150, 380, 5762, 9291, 9830, 293, 3277, 579, 2138, 1919, 2481, 9203, 5343, 5419, 2017, 1752, 7648, 5565, 1700, 3200, 9663, 545, 2082, 959]
Target: 1452
Batch: [4527, 3683, 7136, 0, 1170, 3535, 195, 3285, 2583, 664, 6854, 7150, 7886, 67, 1900, 1793, 2289, 6922, 4288, 4707, 1452, 9613, 9899, 627, 8705, 4670, 9921, 2653, 5867, 6430, 2333, 9593]
Target: 2672
Batch: [5332, 2689, 3609, 2357, 1193, 1439, 7939, 3997, 3763, 4003, 5215, 3577, 885, 535, 764, 9463, 157, 1102, 5419, 9245, 7222, 2672, 5806, 7772, 9849, 98

In [23]:
3978 in test.indices

True

In [40]:
test_dataset.min_m_controlled_sampling(min_symbol=3, batch_size=100)

{4440: [9750,
  9588,
  4782,
  8273,
  5834,
  8409,
  7231,
  1712,
  6643,
  6803,
  1997,
  945,
  4440,
  3579,
  5119,
  8107,
  8867,
  9907,
  4021,
  6171,
  5072,
  3776,
  7203,
  2658,
  7424,
  2885,
  6627,
  1540,
  8558,
  3615,
  1123,
  4998,
  5838,
  5536,
  5510,
  3080,
  1185,
  5437,
  8907,
  7025,
  6696,
  8127,
  2523,
  3384,
  8608,
  4687,
  5498,
  9909,
  6447,
  9138,
  2189,
  4472,
  2099,
  1058,
  1043,
  1047,
  1536,
  5242,
  2147,
  5882,
  8886,
  5009,
  4900,
  4112,
  9523,
  2772,
  5184,
  7131,
  9456,
  861,
  8011,
  3977,
  9430,
  5587,
  145,
  4108,
  1783,
  3362,
  8801,
  5551,
  9337,
  3927,
  4686,
  6742,
  4141,
  2393,
  2684,
  9630,
  139,
  7439,
  3087,
  3875,
  9322,
  8723,
  3712,
  5224,
  2460,
  4828,
  3049,
  9301],
 606: [5086,
  3999,
  2523,
  1973,
  9616,
  2513,
  606,
  6739,
  5551,
  4760,
  7360,
  8835,
  8215,
  5683,
  5134,
  7286,
  368,
  2099,
  5552,
  5732,
  6080,
  7969,
  5961,
  1783,
  

In [222]:
for target, batch in result.items():
    print(SolveMinSym(target, batch))

3
3
3
3
3
3
3
3
3
3


In [ ]:
len(set(result[i][1] for i in range(len(result))))

10

In [ ]:
(3, 5, 6, 5) == (3, 5, 6, 5)

In [ ]:
for target in result[3][0]:
    print(SolveMinSym(target, result[3][0]))

In [ ]:
for i in attribute_combinations([1, 2, 3]):
    print(i)

In [ ]:

import torch
print(len(vectors))  # 10000
print(vectors[:10])  # first 10 vectors
